## Requestung Models for Structured Response.
Models can be requested for Structured Response with Pydantic [Strict], JSON Schema or TypeDict [Lenient]


* Use Pydantic models to get type-safe, structured data from LLMs. 
* This ensures you get exactly the data structure you need.
* Models can be requested to provide their response in a format matching a given schema. 

<img src="../../assets/pydantic.png" width="800" height="400">

In [1]:

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from typing import Literal

load_dotenv()

True

In [ ]:
model = ChatOpenAI(model="gpt-5-nano")

### Simple Schema

In [3]:
class Person(BaseModel):
    """Schema for person information."""
    name: str = Field(description="The person's full name")
    age: int = Field(description="The person's age in years", ge=0, le=100)
    occupation: str = Field(description="The person's job or profession")

In [18]:
structured_model = model.with_structured_output(Person)
text = "John Smith is a 35-year-old software engineer from Seattle."
result = structured_model.invoke(f"Extract person information from: {text}")

# Access typed fields directly
print(f"Name: {result.name}")
print(f"Age: {result.age}")
print(f"Occupation: {result.occupation}")


Name: John Smith
Age: 35
Occupation: software engineer


## Check for Validation on Age field

In [17]:
structured_model = model.with_structured_output(Person)
text = "John Smith is a 135-year-old software engineer from Seattle."
result = structured_model.invoke(f"Extract person information from: {text}")
print(f"Name: {result.name}")
print(f"Age: {result.age}")
print(f"Occupation: {result.occupation}")

Name: John Smith
Age: 1
Occupation: software engineer


In [20]:
structured_model = model.with_structured_output(Person)
text = "John Smith is a 135-year-old software engineer from Seattle."
result = structured_model.invoke(f"Extract person information from: {text}")
print(f"Name: {result.name}")
print(f"Age: {result.age}")
print(f"Occupation: {result.occupation}")

Name: John Smith
Age: 95
Occupation: software engineer


### Complex schema with nested objects

In [6]:
class Address(BaseModel):
    """Address information."""
    street: str = Field(description="Street address")
    city: str = Field(description="City name")
    country: str = Field(description="Country name")

class Company(BaseModel):
    """Company information with nested address."""
    name: str = Field(description="Company name")
    industry: Literal["Technology", "Finance", "Healthcare", "Retail", "Other"] = Field(
        description="Industry sector"
    )
    employee_count: int = Field(description="Number of employees")
    headquarters: Address = Field(description="Company headquarters location")
    is_public: bool = Field(description="Whether the company is publicly traded")

In [7]:
structured_model = model.with_structured_output(Company)

text = """
    NovaPeak Analytics is an artificial intelligence and cloud software company headquartered in Austin, Texas, USA.
    Founded in 2018, the company develops enterprise data analytics platforms and AI-powered automation solutions for retail and healthcare organizations.
    NovaPeak Analytics employs approximately 3,850 people across North America and Europe.
    The company is privately owned and operates regional offices in Toronto, London, and Berlin.
"""

result = structured_model.invoke(f"Extract company information from: {text}")

print(f"Company: {result.name}")
print(f"Industry: {result.industry}")
print(f"Employees: {result.employee_count:,}")
print(f"Location: {result.headquarters.city}, {result.headquarters.country}")
print(f"Public: {result.is_public}")

Company: NovaPeak Analytics
Industry: Technology
Employees: 3,850
Location: Austin, USA
Public: False
